# Two-Tower Gene Interaction Model (TT-GIM)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path(".").resolve()
FEAT_DIR = ROOT / "features" / "external"
MEANS_PATH = ROOT / "data" / "training_data_means.csv"

def norm_gene(g) -> str:
    return str(g).strip().upper()

def load_union_genes(feat_dir: Path) -> list[str]:
    p_npy = feat_dir / "union_genes.npy"
    p_txt = feat_dir / "union_genes.txt"
    if p_npy.exists():
        genes = np.load(p_npy, allow_pickle=True).tolist()
    elif p_txt.exists():
        genes = p_txt.read_text().splitlines()
    else:
        raise FileNotFoundError("Missing union_genes.npy or union_genes.txt in features/external/")
    return [norm_gene(g) for g in genes]

def load_feature_blocks(feat_dir: Path) -> dict[str, np.ndarray]:
    blocks = {
        "genept_m3": np.load(feat_dir / "genept_m3_pca128.npy").astype(np.float32),
        "genept_ada": np.load(feat_dir / "genept_ada_pca128.npy").astype(np.float32),
        "go":        np.load(feat_dir / "go_svd128.npy").astype(np.float32),
        "reactome":  np.load(feat_dir / "reactome_svd128.npy").astype(np.float32),
        "string":    np.load(feat_dir / "string_graph_feats.npy").astype(np.float32),
    }
    return blocks

def stack_blocks(blocks: dict[str, np.ndarray], order: tuple[str, ...]) -> np.ndarray:
    mats = [blocks[k] for k in order]
    n0 = mats[0].shape[0]
    for k, m in zip(order, mats):
        if m.shape[0] != n0:
            raise ValueError(f"Row mismatch: {k} has {m.shape[0]} rows, expected {n0}")
    return np.concatenate(mats, axis=1).astype(np.float32)

def align_genes(
    union_genes: list[str],
    X_union: np.ndarray,
    genes_wanted: list[str],
    fill: str = "mean",  # "mean" or "zeros"
):
    gene2idx = {g: i for i, g in enumerate(union_genes)}
    genes_wanted = [norm_gene(g) for g in genes_wanted]

    d = X_union.shape[1]
    X = np.empty((len(genes_wanted), d), dtype=np.float32)
    found = np.zeros(len(genes_wanted), dtype=bool)

    if fill == "mean":
        fill_vec = X_union.mean(axis=0)
    elif fill == "zeros":
        fill_vec = np.zeros(d, dtype=np.float32)
    else:
        raise ValueError("fill must be 'mean' or 'zeros'")

    missing = []
    for i, g in enumerate(genes_wanted):
        j = gene2idx.get(g)
        if j is None:
            X[i] = fill_vec
            missing.append(g)
        else:
            X[i] = X_union[j]
            found[i] = True

    return X, found, missing

df = pd.read_csv(MEANS_PATH)

gene_cols = [c for c in df.columns if c != "pert_symbol"]
out_genes = [norm_gene(g) for g in gene_cols]  # length 5127

base = df.loc[df["pert_symbol"].astype(str).str.lower() == "non-targeting", gene_cols].iloc[0].to_numpy(np.float32)

tr = df.loc[df["pert_symbol"].astype(str).str.lower() != "non-targeting"].reset_index(drop=True)
train_genes = [norm_gene(g) for g in tr["pert_symbol"].astype(str).tolist()]  # length 80

D_train = tr[gene_cols].to_numpy(np.float32) - base[None, :]  # (80, 5127)

union_genes = load_union_genes(FEAT_DIR)
blocks = load_feature_blocks(FEAT_DIR)

order = ("genept_m3", "genept_ada", "string", "go", "reactome")
X_union = stack_blocks(blocks, order)

print("X_union:", X_union.shape, "| union_genes:", len(union_genes))

X_out, out_found, out_missing = align_genes(union_genes, X_union, out_genes, fill="mean")
X_pert, pert_found, pert_missing = align_genes(union_genes, X_union, train_genes, fill="mean")

print(f"X_out : {X_out.shape} | missing {len(out_missing)} / {len(out_genes)}")
print(f"X_pert: {X_pert.shape} | missing {len(pert_missing)} / {len(train_genes)}")

if pert_missing:
    print("Missing perturbation genes:", pert_missing[:50])

assert X_out.shape[0] == 5127
assert X_pert.shape[0] == D_train.shape[0] == len(train_genes)
assert D_train.shape[1] == 5127


X_union: (5143, 515) | union_genes: 5143
X_out : (5127, 515) | missing 0 / 5127
X_pert: (80, 515) | missing 0 / 80


In [26]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def set_seed(seed=6):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(6)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [27]:
class MLP(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_out),
        )

    def forward(self, x):
        return self.net(x)

In [28]:
class TTGIM(nn.Module):
    """
    Two-Tower Gene Interaction Model:
      p = f_pert(x_pert)      (B, r)
      O = f_out(X_out)        (G, r)
      raw = p @ O.T           (B, G)
      delta = softplus(scale(x_pert)) * raw + gene_bias
    """
    def __init__(self, d_pert, d_out, G, r=64, hidden=256, dropout=0.1):
        super().__init__()
        self.pert_tower = MLP(d_pert, hidden, r, dropout=dropout)
        self.out_tower  = MLP(d_out,  hidden, r, dropout=dropout)

        self.scale_head = nn.Sequential(
            nn.Linear(d_pert, hidden),
            nn.GELU(),
            nn.Linear(hidden, 1),
        )

        self.gene_bias = nn.Parameter(torch.zeros(1, G))

    def forward(self, x_pert, X_out):
        # x_pert: (B, d_pert)
        # X_out : (G, d_out)
        p = self.pert_tower(x_pert)   # (B, r)
        O = self.out_tower(X_out)     # (G, r)
        raw = p @ O.T                 # (B, G)
        scale = F.softplus(self.scale_head(x_pert))  # (B, 1)
        return scale * raw + self.gene_bias

In [29]:
def smoothstep(t):
    return t * t * (3.0 - 2.0 * t)

def gate_smoothstep(x, a=0.0, b=0.2):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return smoothstep(t)

def weighted_mae(y, yhat, w, eps=1e-12):
    num = torch.sum(w * torch.abs(yhat - y), dim=1)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)
    return torch.mean(num / den)

def weighted_cos(y, yhat, w, eps=1e-12):
    wy  = w * y
    why = w * yhat
    num = torch.sum(wy * why, dim=1)
    dy  = torch.sqrt(torch.sum(wy * wy, dim=1))
    dh  = torch.sqrt(torch.sum(why * why, dim=1))
    den = torch.clamp(dy * dh, min=eps)
    return torch.mean(num / den)

def myllia_like_loss(y, yhat, gene_gate_a=0.0, gene_gate_b=0.2, lam=0.7):
    w = gate_smoothstep(torch.abs(y), a=gene_gate_a, b=gene_gate_b)
    l1 = weighted_mae(y, yhat, w)
    cos = weighted_cos(y, yhat, w)
    loss = lam * l1 + (1.0 - lam) * (1.0 - cos)
    return loss, {"wmae": float(l1.detach().cpu()), "wcos": float(cos.detach().cpu())}

In [31]:
from sklearn.model_selection import KFold
from myllia_metric import myllia_score
def train_one_fold(
    X_pert_np, D_np, X_out_np,
    tr_idx, va_idx,
    *,
    r=64,
    hidden=256,
    dropout=0.1,
    lr=1e-3,
    wd=1e-2,
    max_epochs=2000,
    patience=150,
    lam=0.7,
):
    G = D_np.shape[1]
    d = X_pert_np.shape[1]

    X_out = torch.tensor(X_out_np, dtype=torch.float32, device=device)

    model = TTGIM(d_pert=d, d_out=d, G=G, r=r, hidden=hidden, dropout=dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    X_tr = torch.tensor(X_pert_np[tr_idx], dtype=torch.float32, device=device)
    Y_tr = torch.tensor(D_np[tr_idx],      dtype=torch.float32, device=device)
    X_va = torch.tensor(X_pert_np[va_idx], dtype=torch.float32, device=device)
    Y_va = D_np[va_idx].astype(np.float32)

    best = {"score": -1e9, "state": None, "epoch": -1}
    bad = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        Y_hat = model(X_tr, X_out)
        loss, logs = myllia_like_loss(Y_tr, Y_hat, lam=lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if epoch % 10 == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                pred_va = model(X_va, X_out).detach().cpu().numpy().astype(np.float32)

            m = myllia_score(Y_va, pred_va)

            if m.score > best["score"]:
                best["score"] = m.score
                best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best["epoch"] = epoch
                bad = 0
            else:
                bad += 1

            if bad >= patience:
                break

    # restore best
    model.load_state_dict(best["state"])
    model.eval()
    with torch.no_grad():
        pred_va = model(X_va, X_out).detach().cpu().numpy().astype(np.float32)
    m = myllia_score(Y_va, pred_va)

    return model, m, best["epoch"]

def cv_train_ttgim(
    X_pert_np, D_np, X_out_np,
    *,
    n_splits=5,
    seed=6,
    **train_kwargs
):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    fold_metrics = []
    fold_models = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_pert_np), start=1):
        model, m, best_epoch = train_one_fold(
            X_pert_np, D_np, X_out_np, tr_idx, va_idx, **train_kwargs
        )
        fold_models.append(model)
        fold_metrics.append(m)

        print(
            f"[fold {fold}] best_epoch={best_epoch} "
            f"score={m.score:+.5f}  wcos={m.wcos:+.5f}  "
            f"pred_wmae={m.pred_wmae:.5f}  ratio={m.wmae_ratio:.5f}  mean_term={m.mean_term:+.5f}"
        )

    # summary
    scores = np.array([m.score for m in fold_metrics], dtype=np.float32)
    wcos   = np.array([m.wcos  for m in fold_metrics], dtype=np.float32)
    ratio  = np.array([m.wmae_ratio for m in fold_metrics], dtype=np.float32)

    print("\nCV summary:")
    print(f"score: {scores.mean():+.5f} ± {scores.std(ddof=1):.5f}")
    print(f"wcos : {wcos.mean():+.5f} ± {wcos.std(ddof=1):.5f}")
    print(f"ratio: {ratio.mean():.5f} ± {ratio.std(ddof=1):.5f}")

    return fold_models, fold_metrics

In [32]:
train_kwargs = dict(
    r=64,
    hidden=256,
    dropout=0.10,
    lr=1e-3,
    wd=1e-2,
    max_epochs=2000,
    patience=150,
    lam=0.7,
)

fold_models, fold_metrics = cv_train_ttgim(
    X_pert, D_train, X_out,
    n_splits=5,
    seed=6,
    **train_kwargs
)

[fold 1] best_epoch=1830 score=+0.10217  wcos=+0.38232  pred_wmae=0.08115  ratio=0.87653  mean_term=+0.26723
[fold 2] best_epoch=620 score=+0.05626  wcos=+0.38602  pred_wmae=0.08301  ratio=0.92535  mean_term=+0.14574
[fold 3] best_epoch=900 score=+0.12735  wcos=+0.46066  pred_wmae=0.08440  ratio=0.81289  mean_term=+0.27646
[fold 4] best_epoch=1620 score=+0.09024  wcos=+0.40274  pred_wmae=0.07686  ratio=0.89622  mean_term=+0.22406
[fold 5] best_epoch=540 score=+0.03966  wcos=+0.27521  pred_wmae=0.08535  ratio=0.91444  mean_term=+0.14411

CV summary:
score: +0.08314 ± 0.03529
wcos : +0.38139 ± 0.06715
ratio: 0.88509 ± 0.04442


In [33]:
def train_full_model(X_pert_np, D_np, X_out_np, **train_kwargs):
    idx = np.arange(X_pert_np.shape[0])
    model, m, best_epoch = train_one_fold(
        X_pert_np, D_np, X_out_np, idx, idx, **train_kwargs
    )
    print(f"[full] best_epoch={best_epoch} score(train-as-val)={m.score:+.5f} wcos={m.wcos:+.5f} ratio={m.wmae_ratio:.5f}")
    return model

final_model = train_full_model(X_pert, D_train, X_out, **train_kwargs)


[full] best_epoch=1970 score(train-as-val)=+1.85408 wcos=+0.98553 ratio=0.27365


In [34]:
import pandas as pd

VALMAP_PATH = ROOT / "data" / "pert_ids_val.csv"
sub_path = ROOT / "model4_submission.csv"

val_df = pd.read_csv(VALMAP_PATH)
val_map = {f"pert_{int(pid.split('_')[-1])}" if isinstance(pid, str) else pid: pert
           for pert, pid in zip(val_df["pert"].astype(str), val_df["pert_id"].astype(str))}
val_map = {str(r["pert_id"]): str(r["pert"]) for _, r in val_df.iterrows()}

gene2idx_union = {g: i for i, g in enumerate(union_genes)}
mean_feat = X_union.mean(axis=0).astype(np.float32)

def get_feat_for_gene(g: str) -> np.ndarray:
    gg = norm_gene(g)
    j = gene2idx_union.get(gg, None)
    if j is None:
        return mean_feat
    return X_union[j].astype(np.float32)

delta_baseline = D_train.mean(axis=0).astype(np.float32)

X_out_t = torch.tensor(X_out, dtype=torch.float32, device=device)
final_model.eval()

rows = []
for i in range(1, 121):
    pert_id = f"pert_{i}"
    if pert_id in val_map:
        gene = val_map[pert_id]
        x = torch.tensor(get_feat_for_gene(gene)[None, :], dtype=torch.float32, device=device)
        with torch.no_grad():
            d_hat = final_model(x, X_out_t).detach().cpu().numpy().astype(np.float32)[0]
    else:
        d_hat = delta_baseline

    row = {"pert_id": pert_id}
    row.update({g: float(d_hat[j]) for j, g in enumerate(gene_cols)})
    rows.append(row)

sub = pd.DataFrame(rows)
sub.to_csv(sub_path, index=False)
print("Created:", sub_path, "| shape:", sub.shape)

Created: C:\Users\rowes\Documents\GitHub\2026-ML-Projects\Myllia_Competition\model4_submission.csv | shape: (120, 5128)


In [40]:
import pandas as pd
import numpy as np

sub = pd.read_csv("model4_submission.csv")
gene_cols = [c for c in sub.columns if c != "pert_id"]

sub_lb = sub[sub["pert_id"].isin([f"pert_{i}" for i in range(1,61)])]
Y = sub_lb[gene_cols].to_numpy(np.float32)

print("LB preds shape:", Y.shape)
print("Mean std across perts:", float(Y.std(axis=0).mean()))
print("Mean abs value:", float(np.abs(Y).mean()))


LB preds shape: (60, 5127)
Mean std across perts: 0.03490680456161499
Mean abs value: 0.03732050955295563


In [42]:
import numpy as np
import pandas as pd

sub = pd.read_csv("model4_submission.csv")
gene_cols = [c for c in sub.columns if c != "pert_id"]
Y = sub[sub["pert_id"].isin([f"pert_{i}" for i in range(1,61)])][gene_cols].to_numpy(np.float32)

# average pairwise cosine similarity of predicted deltas
Yn = Y / (np.linalg.norm(Y, axis=1, keepdims=True) + 1e-12)
cos = Yn @ Yn.T
mean_offdiag = (cos.sum() - np.trace(cos)) / (cos.size - len(cos))
print("Mean pairwise cosine (off-diag):", float(mean_offdiag))


Mean pairwise cosine (off-diag): 0.5360198020935059


In [44]:
import pandas as pd
from pathlib import Path

ROOT = Path(".").resolve()
sample = pd.read_csv(ROOT / "data" / "sample_submission.csv")
sub = pd.read_csv("model4_submission.csv")

sub = sub[sample.columns]  # exact ordering
sub.to_csv("submission_fixed.csv", index=False)
print("Wrote submission_fixed.csv")


Wrote submission_fixed.csv
